# Cross-Session Alignment Regime Sweep

**Question.** Does eye-affine cross-session alignment tighten spatial distributions of orofacial bodyparts compared to midpoint centering alone?

**Experimental design.** For each animal in the cohort and every session in its recording range:

1. Compute the eye-based similarity transform (session → reference).
2. Apply three alignment regimes to each session's DLC output and compute the resulting 2D histogram for each bodypart:
   - `eye_affine` — similarity transform (translation + rotation + uniform scale) fit to (L_eye, R_eye).
   - `eye_centered` — translate so eye-midpoint aligns to reference (no rotation / scale).
   - `spout_centered` — translate so spout-midpoint aligns to reference (current default for tongue).
3. For each (animal, bodypart, regime), quantify cross-session similarity by computing all pairwise 2D-histogram comparisons and averaging:
   - **Pearson correlation** of flattened normalized histograms (higher = better alignment).
   - **Jensen-Shannon divergence** (lower = better alignment).
4. Visualize pooled heatmaps and per-session eye-fiducial scatters.

**Reference frame.** Per-animal mean of eye-medians across all pre-stroke sessions. Mean-of-baseline is lower-noise than any single session. Flipping to last-pre-stroke-session is a one-line toggle in the USER INPUTS cell.

**Output.** A single summary chart (`00_metric_overview.png`) that shows, per (animal, bodypart), whether any regime is meaningfully better than the others. Plus per-animal detail folders for visual verification.

**Interpretation rule.**
- If `eye_affine` > `eye_centered` > `spout_centered` in mean Pearson r for most (animal, bodypart) combos → **build Phase 4.5**.
- If regimes are indistinguishable (within noise) → **skip Phase 4.5**; midpoint centering is sufficient.
- If `eye_centered` ≈ `eye_affine` > `spout_centered` → the gain is from eye (vs spout) fiducial choice, not from similarity vs translation. Consider a lightweight "Phase 4.5-lite" that only adds eye-midpoint centering.


In [ ]:
# ============================================================
# USER INPUTS — edit, then run all cells.
# ============================================================

from pathlib import Path
import datetime

# Animals to include. Default = full cohort from configs/animals.yaml.
ANIMALS = ["PS46", "PS47", "PS48", "PS49", "PS50", "PS51", "PS54", "PS55"]

# Bodyparts whose cross-session distributions we compare.
# Eyes and spouts are fiducials — excluded from comparison (they're what we align ON).
COMPARE_BODYPARTS = [
    "tongue", "jaw", "nose",
    "L_whiskers_1", "L_whiskers_2", "L_whiskers_3",
    "R_whiskers_1", "R_whiskers_2", "R_whiskers_3",
]

# Likelihood threshold for gating bodypart positions before histogram accumulation.
# 0.7 is the canonical default from the legacy tongue/jaw pipeline.
BODYPART_LIKELIHOOD_THRESHOLD = 0.7

# Eye fiducial likelihood threshold — higher because fiducials must be cleanly detected
# to compute a reliable similarity transform.
EYE_LIKELIHOOD_THRESHOLD = 0.9

# Spout fiducial likelihood threshold for spout_centered regime.
SPOUT_LIKELIHOOD_THRESHOLD = 0.7

# Minimum likelihood-passing frames required per session per bodypart to include
# that (session, bodypart) in the sweep. Prevents nearly-empty histograms from
# polluting the pairwise metrics.
MIN_FRAMES_PER_SESSION = 200

# Reference-frame choice:
#   "mean_of_baseline"  — mean of per-session eye medians across pre-stroke sessions (default)
#   "last_pre_stroke"   — eye medians from the animal's stroke_date session only
REFERENCE_MODE = "mean_of_baseline"

# Heatmap binning.
HEATMAP_NBINS = 80

# Margin (px) added to pooled data range when deriving per-(animal, bodypart) histogram edges.
HEATMAP_MARGIN_PX = 20

# Small epsilon added to histograms before KL/JS divergence (Laplace smoothing).
HIST_EPSILON = 1e-10

# Output location.
_cwd = Path.cwd().resolve()
if _cwd.name == "experiments" and _cwd.parent.name == "notebooks":
    _EXPT_DIR = _cwd
else:
    _EXPT_DIR = _cwd / "notebooks" / "experiments"
OUTPUT_DIR = _EXPT_DIR / "outputs" / "alignment_sweep"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Caching — huge speedup on reruns, crash-resilient.
# ------------------------------------------------------------
# DLC parquet cache: locally-saved parquet copies of each DLC session, kills
# the SMB round-trip on reruns. First run is slow (SMB); subsequent runs read
# from local NVMe. Delete the cache dir to force a fresh pull from MICROSCOPE.
USE_DLC_CACHE = True
DLC_CACHE_DIR = OUTPUT_DIR / "_dlc_cache"

# Per-animal pickle cache: full per-animal sweep result (fiducials, transform
# params, aligned points). A crash mid-run doesn't lose processed animals —
# rerun picks up from the next uncached animal. If a rerun hits a partial
# animal (crashed mid-processing), delete ANIMAL_CACHE_DIR/{animal}.pkl before
# rerunning to force recompute for just that animal.
USE_ANIMAL_CACHE = True
ANIMAL_CACHE_DIR = OUTPUT_DIR / "_animal_cache"

for _cache_dir in (DLC_CACHE_DIR, ANIMAL_CACHE_DIR):
    _cache_dir.mkdir(parents=True, exist_ok=True)
    # Gitignore cache contents — they're large and ephemeral.
    _gi = _cache_dir / ".gitignore"
    if not _gi.exists():
        _gi.write_text("*\n!.gitignore\n")

print(f"Saving outputs to: {OUTPUT_DIR}")
print(f"DLC cache: {DLC_CACHE_DIR} (USE_DLC_CACHE={USE_DLC_CACHE})")
print(f"Animal cache: {ANIMAL_CACHE_DIR} (USE_ANIMAL_CACHE={USE_ANIMAL_CACHE})")
print(f"Animals: {ANIMALS}")
print(f"Bodyparts: {COMPARE_BODYPARTS}")
print(f"Reference mode: {REFERENCE_MODE}")


In [ ]:
import re
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from stroke_orofacial.paths import PathResolver
from stroke_orofacial.animals import load_animals, get_animal
from stroke_orofacial.dlc_io.loaders import discover_dlc_filtered_files, read_dlc

resolver = PathResolver()
animals_dict = load_animals()
print(f"Loaded metadata for {len(animals_dict)} animals")


In [ ]:
# Session discovery: glob the DLC output directory, parse YYYY-MM-DD out of
# "video2_{YYYY-MM-DD}T*_filtered.h5" filenames, return sorted dates.

_FNAME_DATE_RE = re.compile(r"video2_(\d{4}-\d{2}-\d{2})T")


def discover_session_dates(animal: str) -> list[datetime.date]:
    """All session dates with a filtered DLC file for this animal."""
    dlc_dir = resolver.dlc_output_dir(animal)
    dates: set[datetime.date] = set()
    for p in dlc_dir.glob("video2_*T*_filtered.h5"):
        m = _FNAME_DATE_RE.match(p.name)
        if m:
            try:
                dates.add(datetime.date.fromisoformat(m.group(1)))
            except ValueError:
                continue
    return sorted(dates)


def session_dates_for(animal: str) -> list[datetime.date]:
    """Session dates filtered to the animal's recording range and universal exclusions."""
    info = get_animal(animal)
    all_dates = discover_session_dates(animal)
    in_range = [d for d in all_dates if info.session_start <= d <= info.session_end]
    # Drop sessions excluded across all analyses (applies_to contains "all").
    # Exclusion.applies(None) returns True iff the exclusion is universal.
    excluded = {ex.date for ex in info.exclusions if ex.applies(None)}
    return [d for d in in_range if d not in excluded]


# Print per-animal session counts for sanity.
print(f"{'Animal':<8} {'N sessions':<12} {'First':<12} {'Last':<12} {'stroke_date':<12}")
print("-" * 60)
for animal in ANIMALS:
    info = get_animal(animal)
    dates = session_dates_for(animal)
    if dates:
        print(f"{animal:<8} {len(dates):<12} {str(dates[0]):<12} {str(dates[-1]):<12} {str(info.stroke_date):<12}")
    else:
        print(f"{animal:<8} {0:<12} {'—':<12} {'—':<12} {str(info.stroke_date):<12}  WARNING: no sessions found")


In [ ]:
def load_dlc_session(animal: str, date: datetime.date) -> pd.DataFrame | None:
    """Load the filtered DLC H5 for (animal, date). Returns None if not found.

    Caches the loaded DataFrame locally as a parquet on first read, so subsequent
    reads skip the SMB round-trip entirely. Cache lives under DLC_CACHE_DIR.
    """
    cache_path = DLC_CACHE_DIR / f"{animal}_{date.strftime('%Y%m%d')}.parquet"
    if USE_DLC_CACHE and cache_path.exists():
        try:
            return pd.read_parquet(cache_path)
        except Exception as e:
            print(f"    cache read failed for {animal}/{date} ({e}); falling through to SMB")
    dlc_dir = resolver.dlc_output_dir(animal)
    h5_files = discover_dlc_filtered_files(dlc_dir, date.strftime("%Y%m%d"))
    if not h5_files:
        return None
    df = read_dlc(h5_files[0])
    if USE_DLC_CACHE:
        try:
            df.to_parquet(cache_path, compression="zstd")
        except Exception as e:
            print(f"    cache write failed for {animal}/{date} ({e}); continuing without cache")
    return df


def gated_xy(df: pd.DataFrame, bodypart: str, threshold: float) -> np.ndarray:
    """Return (N, 2) array of (x, y) for frames with likelihood >= threshold and finite coords."""
    x = df[f"{bodypart}_x"].to_numpy(dtype=np.float64)
    y = df[f"{bodypart}_y"].to_numpy(dtype=np.float64)
    lik = df[f"{bodypart}_likelihood"].to_numpy(dtype=np.float64)
    mask = (lik >= threshold) & np.isfinite(x) & np.isfinite(y)
    return np.column_stack([x[mask], y[mask]])


def median_xy(df: pd.DataFrame, bodypart: str, threshold: float) -> np.ndarray | None:
    """Median (x, y) for a bodypart after likelihood gating. None if nothing passes."""
    pts = gated_xy(df, bodypart, threshold)
    if len(pts) == 0:
        return None
    return np.median(pts, axis=0)


def median_midpoint(
    df: pd.DataFrame, left_bp: str, right_bp: str, threshold: float
) -> np.ndarray | None:
    """Median of per-frame midpoint of two bodyparts, after both pass likelihood gate."""
    lx = df[f"{left_bp}_x"].to_numpy(dtype=np.float64)
    ly = df[f"{left_bp}_y"].to_numpy(dtype=np.float64)
    rx = df[f"{right_bp}_x"].to_numpy(dtype=np.float64)
    ry = df[f"{right_bp}_y"].to_numpy(dtype=np.float64)
    llik = df[f"{left_bp}_likelihood"].to_numpy(dtype=np.float64)
    rlik = df[f"{right_bp}_likelihood"].to_numpy(dtype=np.float64)
    mask = (
        (llik >= threshold) & (rlik >= threshold)
        & np.isfinite(lx) & np.isfinite(ly)
        & np.isfinite(rx) & np.isfinite(ry)
    )
    if not np.any(mask):
        return None
    mid_x = (lx[mask] + rx[mask]) / 2
    mid_y = (ly[mask] + ry[mask]) / 2
    return np.array([np.median(mid_x), np.median(mid_y)])


In [ ]:
def fit_similarity_2pt(src_pts: np.ndarray, dst_pts: np.ndarray) -> dict:
    """Fit translation + rotation + uniform scale mapping src_pts -> dst_pts.

    Two-point exact fit via complex-number arithmetic.
    Returns dict with scale, theta_rad, tx, ty, matrix M (2x3).
    """
    s = src_pts[:, 0] + 1j * src_pts[:, 1]
    d = dst_pts[:, 0] + 1j * dst_pts[:, 1]
    a = (d[0] - d[1]) / (s[0] - s[1])
    b = d[0] - a * s[0]
    scale = float(np.abs(a))
    theta_rad = float(np.angle(a))
    tx, ty = float(b.real), float(b.imag)
    c, sn = np.cos(theta_rad), np.sin(theta_rad)
    M = np.array([[scale * c, -scale * sn, tx],
                  [scale * sn,  scale * c,  ty]])
    return {"scale": scale, "theta_rad": theta_rad, "tx": tx, "ty": ty, "M": M}


def apply_affine(pts: np.ndarray, M: np.ndarray) -> np.ndarray:
    """Apply 2x3 affine matrix to (N, 2) points."""
    pts_h = np.column_stack([pts, np.ones(len(pts))])
    return (M @ pts_h.T).T


def translate(pts: np.ndarray, shift: np.ndarray) -> np.ndarray:
    """Translate (N, 2) points by a 2-element shift vector."""
    return pts + shift


def apply_transform(pts: np.ndarray, regime: str, transform_entry: dict) -> np.ndarray:
    """Dispatch to the right apply-function for a regime using pure params.

    Takes a transform_entry dict of shape {"params": ...} — no callables — so
    transform_entry is picklable. The dispatch lives here instead of being
    captured in a lambda closure inside compute_session_transforms, so the
    per-animal results dict can be serialized for checkpoint/resume.
    """
    params = transform_entry["params"]
    if regime == "eye_affine":
        return apply_affine(pts, params["M"])
    # eye_centered and spout_centered both use a translation shift.
    return translate(pts, params["shift"])


In [ ]:
def build_reference_frame(animal: str, mode: str = REFERENCE_MODE) -> dict | None:
    """Compute the reference eye + spout fiducial positions for an animal.

    Returns dict with:
      - L_eye_ref, R_eye_ref: (x, y) each — used as similarity-transform targets
      - eye_mid_ref: (x, y) — used by eye_centered regime
      - spout_mid_ref: (x, y) — used by spout_centered regime
      - n_sessions_used: int
      - mode: "mean_of_baseline" or "last_pre_stroke"

    Returns None if the reference cannot be built (e.g., no pre-stroke sessions with
    detectable eyes).
    """
    info = get_animal(animal)
    all_dates = session_dates_for(animal)
    pre_stroke = [d for d in all_dates if d <= info.stroke_date]
    if not pre_stroke:
        print(f"  {animal}: no pre-stroke sessions found")
        return None

    if mode == "last_pre_stroke":
        anchor_dates = [pre_stroke[-1]]
    elif mode == "mean_of_baseline":
        anchor_dates = pre_stroke
    else:
        raise ValueError(f"Unknown REFERENCE_MODE: {mode}")

    Ls, Rs, eye_mids, spout_mids = [], [], [], []
    for d in anchor_dates:
        df = load_dlc_session(animal, d)
        if df is None:
            continue
        l_xy = median_xy(df, "L_eye", EYE_LIKELIHOOD_THRESHOLD)
        r_xy = median_xy(df, "R_eye", EYE_LIKELIHOOD_THRESHOLD)
        if l_xy is None or r_xy is None:
            continue
        Ls.append(l_xy)
        Rs.append(r_xy)
        eye_mids.append((l_xy + r_xy) / 2)
        sp_mid = median_midpoint(df, "L_spout", "R_spout", SPOUT_LIKELIHOOD_THRESHOLD)
        if sp_mid is not None:
            spout_mids.append(sp_mid)

    if len(Ls) == 0:
        print(f"  {animal}: no pre-stroke sessions had detectable eyes")
        return None
    if len(spout_mids) == 0:
        print(f"  {animal}: no pre-stroke sessions had detectable spouts (spout_centered regime will skip this animal)")

    return {
        "L_eye_ref": np.mean(Ls, axis=0),
        "R_eye_ref": np.mean(Rs, axis=0),
        "eye_mid_ref": np.mean(eye_mids, axis=0),
        "spout_mid_ref": np.mean(spout_mids, axis=0) if spout_mids else None,
        "n_sessions_used": len(Ls),
        "mode": mode,
        "anchor_dates": anchor_dates,
    }


In [ ]:
def per_session_fiducials(df: pd.DataFrame) -> dict | None:
    """Compute this session's fiducial medians. None if eyes not detectable."""
    L = median_xy(df, "L_eye", EYE_LIKELIHOOD_THRESHOLD)
    R = median_xy(df, "R_eye", EYE_LIKELIHOOD_THRESHOLD)
    if L is None or R is None:
        return None
    eye_mid = (L + R) / 2
    spout_mid = median_midpoint(df, "L_spout", "R_spout", SPOUT_LIKELIHOOD_THRESHOLD)
    return {"L_eye": L, "R_eye": R, "eye_mid": eye_mid, "spout_mid": spout_mid}


def compute_session_transforms(
    session_fid: dict, reference: dict
) -> dict:
    """Build the three alignment transforms for this session, as pure params dicts.

    Returns dict mapping regime_name -> {"params": ...}.
    No callables stored — apply via apply_transform(pts, regime, entry) so the
    returned structure is picklable for checkpoint/resume.

    If spout_mid is None for the session or reference, spout_centered is returned as None.
    """
    transforms: dict[str, object] = {}

    # (1) eye_affine: similarity transform mapping session eyes to reference eyes.
    src = np.array([session_fid["L_eye"], session_fid["R_eye"]])
    dst = np.array([reference["L_eye_ref"], reference["R_eye_ref"]])
    sim = fit_similarity_2pt(src, dst)
    transforms["eye_affine"] = {"params": sim}

    # (2) eye_centered: translate so session eye midpoint -> reference eye midpoint.
    shift_eye = reference["eye_mid_ref"] - session_fid["eye_mid"]
    transforms["eye_centered"] = {"params": {"shift": shift_eye}}

    # (3) spout_centered: translate so session spout midpoint -> reference spout midpoint.
    if session_fid["spout_mid"] is not None and reference["spout_mid_ref"] is not None:
        shift_spout = reference["spout_mid_ref"] - session_fid["spout_mid"]
        transforms["spout_centered"] = {"params": {"shift": shift_spout}}
    else:
        transforms["spout_centered"] = None

    return transforms


REGIME_ORDER = ["eye_affine", "eye_centered", "spout_centered"]
REGIME_COLORS = {"eye_affine": "tab:blue", "eye_centered": "tab:orange", "spout_centered": "tab:green"}


In [ ]:
# Per-animal rendering + main incremental sweep.
#
# Processes ONE animal at a time end-to-end: compute → render figures → checkpoint.
# Each animal's outputs land on disk before the next animal starts, so a crash
# mid-run preserves whatever animals completed. Reruns skip already-processed
# animals (pickle cache hit) and skip already-loaded DLC sessions (parquet cache hit).

import pickle


# ------------------------------------------------------------
# Per-animal rendering helpers (called inside the main loop below).
# ------------------------------------------------------------

def save_eye_scatters(animal: str, ar: dict, outdir: Path) -> None:
    ref = ar["reference"]
    sessions = ar["sessions"]
    # BEFORE alignment
    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    for d, s in sessions.items():
        ax.scatter(*s["fid"]["L_eye"], c="tab:blue", s=40, edgecolor="black", alpha=0.6)
        ax.scatter(*s["fid"]["R_eye"], c="tab:red", s=40, edgecolor="black", alpha=0.6)
    ax.scatter(*ref["L_eye_ref"], c="tab:blue", s=300, marker="*", edgecolor="black", label="L_eye ref (mean baseline)")
    ax.scatter(*ref["R_eye_ref"], c="tab:red", s=300, marker="*", edgecolor="black", label="R_eye ref (mean baseline)")
    ax.set_aspect("equal"); ax.invert_yaxis()
    ax.set_title(f"{animal} — Per-session eye medians (BEFORE alignment)")
    ax.set_xlabel("x (px)"); ax.set_ylabel("y (px)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(outdir / "eye_scatter_before.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # AFTER eye_affine alignment
    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    for d, s in sessions.items():
        src = np.array([s["fid"]["L_eye"], s["fid"]["R_eye"]])
        aligned = apply_transform(src, "eye_affine", s["transforms"]["eye_affine"])
        ax.scatter(aligned[0, 0], aligned[0, 1], c="tab:blue", s=40, edgecolor="black", alpha=0.6)
        ax.scatter(aligned[1, 0], aligned[1, 1], c="tab:red", s=40, edgecolor="black", alpha=0.6)
    ax.scatter(*ref["L_eye_ref"], c="tab:blue", s=300, marker="*", edgecolor="black", label="L_eye ref")
    ax.scatter(*ref["R_eye_ref"], c="tab:red", s=300, marker="*", edgecolor="black", label="R_eye ref")
    ax.set_aspect("equal"); ax.invert_yaxis()
    ax.set_title(f"{animal} — Per-session eye medians AFTER eye_affine alignment")
    ax.set_xlabel("x (px)"); ax.set_ylabel("y (px)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(outdir / "eye_scatter_after.png", dpi=150, bbox_inches="tight")
    plt.close(fig)


def save_transforms_txt(animal: str, ar: dict, outdir: Path) -> None:
    lines = [f"=== {animal} per-session similarity transforms ===",
             f"Reference mode: {ar['reference']['mode']}",
             f"Reference L_eye: {ar['reference']['L_eye_ref']}",
             f"Reference R_eye: {ar['reference']['R_eye_ref']}",
             f"Reference eye_mid: {ar['reference']['eye_mid_ref']}",
             f"Reference spout_mid: {ar['reference']['spout_mid_ref']}",
             "",
             f"{'Date':<12} {'day_rel':<8} {'scale':<10} {'rot (deg)':<12} {'tx (px)':<10} {'ty (px)':<10}",
             "-" * 70]
    for d in sorted(ar["sessions"].keys()):
        s = ar["sessions"][d]
        sim = s["transforms"]["eye_affine"]["params"]
        lines.append(
            f"{str(d):<12} {s['day_rel']:<+8d} {sim['scale']:<10.5f} "
            f"{float(np.degrees(sim['theta_rad'])):<+12.5f} "
            f"{sim['tx']:<+10.3f} {sim['ty']:<+10.3f}"
        )
    (outdir / "transforms.txt").write_text("\n".join(lines))


def save_pooled_heatmap_grid(animal: str, ar: dict, outdir: Path) -> None:
    """One figure per animal: bodyparts as rows, regimes as columns."""
    bodyparts = COMPARE_BODYPARTS
    regimes = REGIME_ORDER
    fig, axes = plt.subplots(
        len(bodyparts), len(regimes),
        figsize=(4 * len(regimes), 3.2 * len(bodyparts)),
        squeeze=False,
    )
    for i, bp in enumerate(bodyparts):
        for j, regime in enumerate(regimes):
            ax = axes[i, j]
            pts_by_date = ar["aligned_points"][bp][regime]
            if len(pts_by_date) < 2:
                ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
                ax.set_title(f"{bp} / {regime}", fontsize=9)
                ax.set_xticks([]); ax.set_yticks([])
                continue
            all_pts = np.concatenate(list(pts_by_date.values()), axis=0)
            xlim = (all_pts[:, 0].min() - HEATMAP_MARGIN_PX, all_pts[:, 0].max() + HEATMAP_MARGIN_PX)
            ylim = (all_pts[:, 1].min() - HEATMAP_MARGIN_PX, all_pts[:, 1].max() + HEATMAP_MARGIN_PX)
            H, xe, ye = np.histogram2d(
                all_pts[:, 0], all_pts[:, 1],
                bins=HEATMAP_NBINS, range=[xlim, ylim],
            )
            ax.imshow(
                H.T, origin="upper",
                extent=[xe[0], xe[-1], ye[-1], ye[0]],
                cmap="viridis", norm="log" if H.max() > 1 else None,
                aspect="auto",
            )
            ax.set_title(f"{bp} / {regime}  (N={len(pts_by_date)})", fontsize=9)
            ax.set_xlabel("x"); ax.set_ylabel("y")
    fig.suptitle(f"{animal} — pooled heatmaps (all dates)", fontsize=12)
    plt.tight_layout()
    plt.savefig(outdir / "pooled_heatmaps.png", dpi=140, bbox_inches="tight")
    plt.close(fig)


def render_animal_outputs(animal: str, ar: dict) -> None:
    """Render all per-animal outputs. Idempotent — safe to call on cache hits."""
    outdir = OUTPUT_DIR / animal
    outdir.mkdir(parents=True, exist_ok=True)
    save_eye_scatters(animal, ar, outdir)
    save_transforms_txt(animal, ar, outdir)
    save_pooled_heatmap_grid(animal, ar, outdir)


# ------------------------------------------------------------
# Per-animal compute (isolated so the main loop is readable).
# ------------------------------------------------------------

def compute_animal_result(animal: str, reference: dict) -> dict:
    """Load all DLC sessions for this animal and compute transforms + aligned points."""
    info = get_animal(animal)
    dates = session_dates_for(animal)

    ar = {
        "reference": reference,
        "sessions": {},  # date -> {"fid": ..., "transforms": ..., "day_rel": int}
        "aligned_points": {
            bp: {reg: {} for reg in REGIME_ORDER}
            for bp in COMPARE_BODYPARTS
        },  # [bp][regime][date] -> (N, 2) array
    }
    for d in dates:
        df = load_dlc_session(animal, d)
        if df is None:
            continue
        fid = per_session_fiducials(df)
        if fid is None:
            print(f"    {d}: skip (no eye fiducials)")
            continue
        tf = compute_session_transforms(fid, reference)
        ar["sessions"][d] = {
            "fid": fid,
            "transforms": tf,
            "day_rel": info.day_relative_to_stroke(d),
        }
        for bp in COMPARE_BODYPARTS:
            raw_pts = gated_xy(df, bp, BODYPART_LIKELIHOOD_THRESHOLD)
            if len(raw_pts) < MIN_FRAMES_PER_SESSION:
                continue
            for regime in REGIME_ORDER:
                t = tf[regime]
                if t is None:
                    continue
                ar["aligned_points"][bp][regime][d] = apply_transform(raw_pts, regime, t)
    return ar


# ------------------------------------------------------------
# Main loop: one animal at a time, end-to-end.
# ------------------------------------------------------------

results: dict[str, dict] = {}

for animal in ANIMALS:
    print(f"\n=== {animal} ===")

    animal_cache_path = ANIMAL_CACHE_DIR / f"{animal}.pkl"
    if USE_ANIMAL_CACHE and animal_cache_path.exists():
        try:
            with open(animal_cache_path, "rb") as f:
                ar = pickle.load(f)
            print(f"  LOADED FROM CACHE ({animal_cache_path.name}) — {len(ar['sessions'])} sessions")
            # Re-render (cheap; overwrites idempotently so figures stay in sync with cache).
            render_animal_outputs(animal, ar)
            print(f"  re-rendered per-animal outputs")
            results[animal] = ar
            continue
        except Exception as e:
            print(f"  cache load failed ({e}); recomputing")

    reference = build_reference_frame(animal, mode=REFERENCE_MODE)
    if reference is None:
        print(f"  SKIP — no usable reference frame")
        continue
    print(f"  Reference: n_baseline={reference['n_sessions_used']}")

    ar = compute_animal_result(animal, reference)
    print(f"  Computed {len(ar['sessions'])} sessions with full fiducials")

    # Render FIRST — so even if the checkpoint write fails, the figures are on disk.
    render_animal_outputs(animal, ar)
    print(f"  wrote per-animal outputs under {OUTPUT_DIR / animal}")

    # Checkpoint the in-memory state for resume-on-rerun.
    if USE_ANIMAL_CACHE:
        try:
            with open(animal_cache_path, "wb") as f:
                pickle.dump(ar, f)
            print(f"  CHECKPOINTED to {animal_cache_path.name}")
        except Exception as e:
            print(f"  checkpoint write failed ({e}); continuing in-memory")

    results[animal] = ar

print(f"\nSweep done. Results for {len(results)} animals.")


In [ ]:
def pooled_range(points_by_date: dict, margin: float) -> tuple[tuple[float, float], tuple[float, float]]:
    """Compute (xlim, ylim) covering all points across all dates, with margin."""
    all_pts = np.concatenate(list(points_by_date.values()), axis=0)
    xlo, xhi = float(all_pts[:, 0].min()) - margin, float(all_pts[:, 0].max()) + margin
    ylo, yhi = float(all_pts[:, 1].min()) - margin, float(all_pts[:, 1].max()) + margin
    return (xlo, xhi), (ylo, yhi)


def normalize_hist(h: np.ndarray, eps: float = HIST_EPSILON) -> np.ndarray:
    """Normalize to a probability distribution (sum=1) with Laplace smoothing."""
    h = h + eps
    return h / h.sum()


def pairwise_pearson_mean(hist_stack: np.ndarray) -> float:
    """Mean pairwise Pearson r of a stack of flattened histograms (N_sessions, N_bins)."""
    flat = hist_stack.reshape(hist_stack.shape[0], -1)
    corr = np.corrcoef(flat)  # (N, N)
    iu = np.triu_indices(corr.shape[0], k=1)
    return float(np.nanmean(corr[iu]))


def js_divergence_pair(p: np.ndarray, q: np.ndarray) -> float:
    """Jensen-Shannon divergence between two distributions (bits)."""
    m = 0.5 * (p + q)
    def kl(a, b):
        mask = a > 0
        return float(np.sum(a[mask] * np.log2(a[mask] / b[mask])))
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


def pairwise_js_mean(hist_stack: np.ndarray) -> float:
    """Mean pairwise Jensen-Shannon divergence (bits). Lower = more similar."""
    flat = hist_stack.reshape(hist_stack.shape[0], -1)
    N = flat.shape[0]
    vals = []
    for i, j in combinations(range(N), 2):
        vals.append(js_divergence_pair(flat[i], flat[j]))
    return float(np.nanmean(vals)) if vals else float("nan")


# Compute per-(animal, bodypart, regime) metrics.
# Uses a per-(animal, bodypart) common histogram grid — shared across regimes so
# direct comparison is fair. Grid derived from pooled points across regimes.

metric_table: list[dict] = []

for animal, ar in results.items():
    for bp in COMPARE_BODYPARTS:
        # Need at least 2 sessions with points under some regime to compute pairwise metrics.
        all_pts_by_regime = {
            reg: ar["aligned_points"][bp][reg]
            for reg in REGIME_ORDER
            if len(ar["aligned_points"][bp][reg]) >= 2
        }
        if not all_pts_by_regime:
            continue

        # Per-regime histogram grid: derive from that regime's pooled points, NOT
        # a common grid — because each regime's reference-frame origin is different
        # (eye_mid vs spout_mid vs canonical_eye_mid), so shared axes would be
        # apples-to-oranges. Within a regime, though, the grid IS shared across dates,
        # which is what the pairwise metrics need.
        for regime, pts_by_date in all_pts_by_regime.items():
            xlim, ylim = pooled_range(pts_by_date, HEATMAP_MARGIN_PX)
            xedges = np.linspace(xlim[0], xlim[1], HEATMAP_NBINS + 1)
            yedges = np.linspace(ylim[0], ylim[1], HEATMAP_NBINS + 1)
            hist_stack = []
            for d in sorted(pts_by_date.keys()):
                H, _, _ = np.histogram2d(
                    pts_by_date[d][:, 0], pts_by_date[d][:, 1],
                    bins=[xedges, yedges],
                )
                hist_stack.append(normalize_hist(H))
            hs = np.stack(hist_stack, axis=0)
            metric_table.append({
                "animal": animal,
                "bodypart": bp,
                "regime": regime,
                "n_sessions": hs.shape[0],
                "pearson_mean": pairwise_pearson_mean(hs),
                "js_mean_bits": pairwise_js_mean(hs),
                "xlim": xlim,
                "ylim": ylim,
                "xedges": xedges,
                "yedges": yedges,
                "hist_stack": hs,
            })

metric_df = pd.DataFrame([
    {k: v for k, v in row.items() if k not in ("xedges", "yedges", "hist_stack")}
    for row in metric_table
])
print(f"Computed {len(metric_df)} (animal, bodypart, regime) metrics")
print(metric_df[["animal", "bodypart", "regime", "n_sessions", "pearson_mean", "js_mean_bits"]].head(15))


In [ ]:
# Cross-animal summary: for each metric, a grid of (animal x bodypart) subplots,
# each subplot showing three bars (one per regime).
# Two figures: one for pearson_mean, one for js_mean_bits.

def save_metric_overview(metric_key: str, higher_is_better: bool, outpath: Path) -> None:
    animals_seen = sorted(set(row["animal"] for row in metric_table))
    bodyparts = COMPARE_BODYPARTS
    # Layout: rows=bodyparts, cols=animals
    fig, axes = plt.subplots(
        len(bodyparts), len(animals_seen),
        figsize=(1.8 * len(animals_seen), 2.0 * len(bodyparts)),
        sharey="row", squeeze=False,
    )
    regime_labels = REGIME_ORDER
    x = np.arange(len(regime_labels))
    for i, bp in enumerate(bodyparts):
        for j, animal in enumerate(animals_seen):
            ax = axes[i, j]
            vals = []
            for reg in regime_labels:
                matches = [r for r in metric_table
                           if r["animal"] == animal and r["bodypart"] == bp and r["regime"] == reg]
                vals.append(matches[0][metric_key] if matches else np.nan)
            bars = ax.bar(x, vals, color=[REGIME_COLORS[r] for r in regime_labels])
            ax.set_xticks(x)
            ax.set_xticklabels(regime_labels, rotation=60, fontsize=6)
            if i == 0:
                ax.set_title(animal, fontsize=9)
            if j == 0:
                ax.set_ylabel(bp, fontsize=8)
            ax.grid(axis="y", alpha=0.3)
            # Highlight best bar
            finite_vals = [v for v in vals if np.isfinite(v)]
            if finite_vals:
                best = max(finite_vals) if higher_is_better else min(finite_vals)
                for bar, v in zip(bars, vals):
                    if np.isfinite(v) and abs(v - best) < 1e-12:
                        bar.set_edgecolor("black")
                        bar.set_linewidth(1.5)
    direction = "higher = better" if higher_is_better else "lower = better"
    fig.suptitle(f"{metric_key} per (animal, bodypart, regime) — {direction}", fontsize=11)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150, bbox_inches="tight")
    plt.close(fig)


save_metric_overview("pearson_mean", higher_is_better=True,
                     outpath=OUTPUT_DIR / "00_metric_overview_pearson.png")
save_metric_overview("js_mean_bits", higher_is_better=False,
                     outpath=OUTPUT_DIR / "00_metric_overview_js_divergence.png")
print("Wrote metric overview figures.")


In [ ]:
# Cross-animal transform summary: histogram of scale, rotation, translation
# magnitudes, colored by animal (and optionally by day_relative_to_stroke).

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

scales_by_animal = {}
rots_by_animal = {}
tmags_by_animal = {}
for animal, ar in results.items():
    scales_by_animal[animal] = []
    rots_by_animal[animal] = []
    tmags_by_animal[animal] = []
    for d, s in ar["sessions"].items():
        sim = s["transforms"]["eye_affine"]["params"]
        scales_by_animal[animal].append(sim["scale"])
        rots_by_animal[animal].append(float(np.degrees(sim["theta_rad"])))
        tmags_by_animal[animal].append(float(np.hypot(sim["tx"], sim["ty"])))

# (0,0) scale distribution per animal (box/swarm)
ax = axes[0, 0]
data = [scales_by_animal[a] for a in ANIMALS if a in scales_by_animal]
labels = [a for a in ANIMALS if a in scales_by_animal]
ax.boxplot(data, labels=labels)
ax.axhline(1.0, color="black", linestyle="--", linewidth=0.7)
ax.set_title("Scale factor per session")
ax.set_ylabel("scale (1.0 = no zoom drift)")
ax.grid(alpha=0.3)

# (0,1) rotation per animal
ax = axes[0, 1]
data = [rots_by_animal[a] for a in ANIMALS if a in rots_by_animal]
ax.boxplot(data, labels=labels)
ax.axhline(0.0, color="black", linestyle="--", linewidth=0.7)
ax.set_title("Rotation per session (deg)")
ax.set_ylabel("rotation (deg)")
ax.grid(alpha=0.3)

# (1,0) translation magnitude per animal
ax = axes[1, 0]
data = [tmags_by_animal[a] for a in ANIMALS if a in tmags_by_animal]
ax.boxplot(data, labels=labels)
ax.set_title("Translation magnitude per session (px)")
ax.set_ylabel("|t| (px)")
ax.grid(alpha=0.3)

# (1,1) scale vs day_relative_to_stroke — look for drift over time
ax = axes[1, 1]
for animal, ar in results.items():
    xs = [s["day_rel"] for s in ar["sessions"].values()]
    ys = [s["transforms"]["eye_affine"]["params"]["scale"] for s in ar["sessions"].values()]
    ax.plot(xs, ys, "o-", markersize=3, alpha=0.6, label=animal)
ax.axhline(1.0, color="black", linestyle="--", linewidth=0.7)
ax.axvline(0.0, color="red", linestyle="--", linewidth=0.7, alpha=0.6)
ax.set_title("Scale factor vs day relative to stroke")
ax.set_xlabel("day relative to stroke (0 = stroke_date)")
ax.set_ylabel("scale")
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_transform_summary.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Wrote transform summary figure.")


In [ ]:
# Write a plain-text summary for interpretation.

lines = [
    "=== Cross-Session Alignment Regime Sweep ===",
    f"Reference mode: {REFERENCE_MODE}",
    f"Animals included: {list(results.keys())}",
    f"Bodyparts compared: {COMPARE_BODYPARTS}",
    f"Regimes: {REGIME_ORDER}",
    f"Likelihood thresholds: bodypart={BODYPART_LIKELIHOOD_THRESHOLD}, eye={EYE_LIKELIHOOD_THRESHOLD}, spout={SPOUT_LIKELIHOOD_THRESHOLD}",
    f"Heatmap bins: {HEATMAP_NBINS} x {HEATMAP_NBINS}",
    "",
    "== Per-animal transform ranges (eye_affine) ==",
    f"{'Animal':<8} {'N':<5} {'scale_mean':<12} {'scale_std':<12} {'rot_std_deg':<14} {'|t|_median_px':<14} {'|t|_max_px':<12}",
    "-" * 85,
]
for animal, ar in results.items():
    scales = [s["transforms"]["eye_affine"]["params"]["scale"] for s in ar["sessions"].values()]
    rots = [float(np.degrees(s["transforms"]["eye_affine"]["params"]["theta_rad"])) for s in ar["sessions"].values()]
    tmags = [float(np.hypot(s["transforms"]["eye_affine"]["params"]["tx"],
                             s["transforms"]["eye_affine"]["params"]["ty"])) for s in ar["sessions"].values()]
    lines.append(
        f"{animal:<8} {len(scales):<5} {np.mean(scales):<12.5f} {np.std(scales):<12.5f} "
        f"{np.std(rots):<14.4f} {np.median(tmags):<14.3f} {np.max(tmags):<12.3f}"
    )

lines += [
    "",
    "== Metric table (pairwise across sessions, aggregated) ==",
    f"{'Animal':<8} {'Bodypart':<16} {'Regime':<16} {'N_sess':<8} {'Pearson r':<12} {'JS (bits)':<12}",
    "-" * 85,
]
for row in metric_table:
    lines.append(
        f"{row['animal']:<8} {row['bodypart']:<16} {row['regime']:<16} {row['n_sessions']:<8} "
        f"{row['pearson_mean']:<12.4f} {row['js_mean_bits']:<12.4f}"
    )

# Per-bodypart regime ranking
lines += ["", "== Per-bodypart regime ranking (Pearson: higher wins; JS: lower wins) =="]
for bp in COMPARE_BODYPARTS:
    rows = [r for r in metric_table if r["bodypart"] == bp]
    if not rows:
        continue
    lines.append(f"\n  Bodypart: {bp}")
    # Aggregate across animals per regime
    regime_to_pearsons: dict[str, list[float]] = {reg: [] for reg in REGIME_ORDER}
    regime_to_js: dict[str, list[float]] = {reg: [] for reg in REGIME_ORDER}
    for r in rows:
        if np.isfinite(r["pearson_mean"]):
            regime_to_pearsons[r["regime"]].append(r["pearson_mean"])
        if np.isfinite(r["js_mean_bits"]):
            regime_to_js[r["regime"]].append(r["js_mean_bits"])
    for reg in REGIME_ORDER:
        ps = regime_to_pearsons[reg]
        js = regime_to_js[reg]
        if ps and js:
            lines.append(
                f"    {reg:<16}  mean pearson r = {np.mean(ps):.4f} (N_animals={len(ps)})  "
                f"mean JS = {np.mean(js):.4f} bits"
            )

(OUTPUT_DIR / "summary.txt").write_text("\n".join(lines))
print(f"Wrote {OUTPUT_DIR / 'summary.txt'}")
print("\nAll outputs written to", OUTPUT_DIR)


## Interpretation checklist

1. **`01_transform_summary.png`** — do scale / rotation / translation magnitudes exceed the thresholds that would justify a similarity transform?
   - Scale std > 1% of mean across sessions → zoom drift is real.
   - Rotation std > 0.5° → tilt drift is real.
   - Translation magnitudes of several px are expected; midpoint centering already handles them.
   - **Look at the bottom-right subplot**: scale vs day-relative-to-stroke. A systematic scale-change around day 0 would indicate the camera was remounted around stroke surgery (common).

2. **`00_metric_overview_pearson.png` + `00_metric_overview_js_divergence.png`** — within each (animal, bodypart) cell, which regime has the darkest-outlined (winning) bar?
   - If `eye_affine` wins for most cells → **Phase 4.5 is worth building**.
   - If `eye_centered` ≈ `eye_affine` and both beat `spout_centered` → **Phase 4.5-lite**: just add eye-midpoint centering, skip the rotation/scale machinery.
   - If regimes are indistinguishable → **skip Phase 4.5**; midpoint centering is sufficient.

3. **Per-animal `pooled_heatmaps.png`** — visual sanity-check. Do the columns (regimes) look meaningfully different for the same bodypart? If not, the metric should confirm.

4. **Per-animal `eye_scatter_before.png` / `eye_scatter_after.png`** — did the transform collapse all per-session eye medians onto the reference stars? If yes, the math is fitting correctly.

5. **Watch for post-stroke anomalies** — an animal where eye_affine works well pre-stroke but diverges post-stroke would indicate that eyes ARE shifting post-stroke (contradicts our a priori assumption). If that happens, the experiment is telling us to find different fiducials or restrict the transform window to pre-stroke sessions.

**To decide**: once results are in, share summary.txt + the metric overview PNGs with the decision notes below captured in `docs/DECISIONS.md`. Either propose a Phase 4.5 spec or close the observation as "midpoint centering sufficient".
